# 10. 프로그램 구조화 예제

## Goal

- 입력·검증·처리·출력 책임을 분리합니다.
- 입출력 없이 핵심 로직을 테스트합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

작은 이벤트 목록을 메모리에서 처리합니다.


## Steps

### 계층별 함수 조립

각 함수가 한 가지 책임만 맡고 상위 함수가 흐름을 연결합니다.


In [1]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Event:
    action: str
    source_ip: str


def validate(raw):
    action = raw.get("action", "").upper()
    source_ip = raw.get("source_ip", "")
    if action not in {"ALLOW", "DENY"} or not source_ip:
        raise ValueError("이벤트 형식 오류")
    return Event(action, source_ip)


def summarize(events):
    return {action: sum(event.action == action for event in events) for action in ("ALLOW", "DENY")}


def render(summary):
    return f"ALLOW={summary['ALLOW']} DENY={summary['DENY']}"


def run_pipeline(raw_records):
    events = [validate(record) for record in raw_records]
    return render(summarize(events))


pipeline_output = run_pipeline([
    {"action": "allow", "source_ip": "192.0.2.10"},
    {"action": "DENY", "source_ip": "198.51.100.20"},
])
print(pipeline_output)


ALLOW=1 DENY=1


## Checks

핵심 처리 함수와 전체 파이프라인 결과를 각각 확인합니다.


In [2]:
assert summarize([Event("DENY", "192.0.2.1")]) == {"ALLOW": 0, "DENY": 1}
assert pipeline_output == "ALLOW=1 DENY=1"
try:
    validate({"action": "UNKNOWN", "source_ip": "192.0.2.1"})
except ValueError:
    print("계약 위반 거부 확인")


계약 위반 거부 확인


## Next Steps

실제 패키지에서는 이 함수들을 모듈로 나누고 공개 API와 CLI 진입점을 문서화합니다.
